In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
base_raw_path = "/Volumes/capstone/bronze/raw/files"
transactions_path = f"{base_raw_path}/landing/transactions_raw"
customers_path = f"{base_raw_path}/landing/customers_raw"
products_path = f"{base_raw_path}/landing/products_raw"
checkpointlocation = f"{base_raw_path}/checkpoint"

In [0]:
transactions_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("item_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("price", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("corrupted_flag", StringType(), True)
])

In [0]:
# customers_schema = "customer_id STRING, name STRING, contact MAP<STRING,STRING>, region STRING"
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("contact", StructType([
        StructField("email", StringType(), True)
    ]), True),
    StructField("region", StringType(), True)
])


In [0]:
products_schema = StructType([
    StructField("item_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True)
])

In [0]:
# Read from the Source Transaction Folder
bronze_txn_df = (spark.readStream
                     .format("cloudFiles")
                     .option("cloudFiles.format", "csv")
                     .option("header", "true")
                     .schema(transactions_schema)
                     .load(transactions_path))

# Add additional columns for ingestion timestamp and source file name
bronze_txn_write = (bronze_txn_df
                    .withColumn("_ingest_timestamp", current_timestamp())
                    .withColumn("_source_file_name",col("_metadata.file_path")))

# Create the Delta Table
(bronze_txn_write.writeStream
    .format("delta")
    .option("checkpointLocation", checkpointlocation + "/bronze_transaction")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(once=True)
    .toTable("capstone.bronze.transactions"))

In [0]:
%sql

select * from capstone.bronze.transactions;

In [0]:
bronze_prod_df = (spark.readStream
                      .format("cloudFiles")
                      .option("cloudFiles.format", "csv")
                      .option("header", "true")
                      .schema(products_schema)
                      .load(products_path))

bronze_prod_write = bronze_prod_df.withColumn("_ingest_timestamp", current_timestamp()) \
                                  .withColumn("_source_file_name", col("_metadata.file_path"))
                                  
(bronze_prod_write.writeStream
    .format("delta")
    .option("checkpointLocation", checkpointlocation + "/bronze_products")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(once=True)
    .toTable("capstone.bronze.products"))

In [0]:
%sql

select * from capstone.bronze.products

In [0]:
bronze_cust_df = (spark.readStream
                      .format("cloudFiles")
                      .option("cloudFiles.format", "json")
                      .option("multiline", "true")
                      .schema(customers_schema)
                      .load(customers_path))

bronze_cust_write = bronze_cust_df.withColumn("_ingest_timestamp", current_timestamp()) \
                                    .withColumn("_source_file_name", col("_metadata.file_path")) 

(bronze_cust_write.writeStream
    .format("delta")
    .option("checkpointLocation", checkpointlocation + "/bronze_customers1")
    .outputMode("append")
    .option("mergeSchema", "true")
    .trigger(once=True)
    .toTable("capstone.bronze.customers"))

In [0]:
%sql
SELECT * FROM capstone.bronze.customers;

In [0]:
print("Bronze Table Counts:")
print("capstone.bronze.transactions:", spark.table("capstone.bronze.transactions").count())
print("capstone.bronze.customers:", spark.table("capstone.bronze.customers").count())
print("capstone.bronze.products:", spark.table("capstone.bronze.products").count())


In [0]:
# %sql
# DROP TABLE capstone.bronze.transactions;
# DROP TABLE capstone.bronze.customers;
# DROP TABLE capstone.bronze.products;